# Augmentation and robustness

Measure model sensitivity to controlled corruptions without changing labels.

Run with the repository virtual environment. These are synthetic research experiments; units and assumptions are explicit below.

## Inputs

Transformations are seeded and preserve the image/mask coordinate contract.

In [ ]:
import matplotlib.pyplot as plt

from autonomous_vehicles.data import RoadScenes
from autonomous_vehicles.evaluation.robustness import evaluate_corruptions
from autonomous_vehicles.experiments.reproducibility import reproducible
from autonomous_vehicles.model import RoadNet
from autonomous_vehicles.perception.augmentation import (
    AugmentationConfig,
    Augmenter,
    image_corruption,
)

image, mask = RoadScenes(size=32)[0]

## Aligned augmentation

Horizontal flips apply to both input and target. Color transforms apply only to the input.

In [ ]:
augmenter = Augmenter(AugmentationConfig(), seed=42)
fig, axes = plt.subplots(2, 4, figsize=(11, 5))
for column in range(4):
    transformed, labels = augmenter(image, mask)
    axes[0, column].imshow(transformed.permute(1, 2, 0))
    axes[1, column].imshow(labels, vmin=0, vmax=2)
    assert transformed.min() >= 0 and transformed.max() <= 1
plt.tight_layout()
plt.show()

## Corruption grid

Severity is a controlled parameter, not a standardized real-world weather scale.

In [ ]:
kinds = ("darkness", "fog", "noise", "occlusion")
severities = (0, 0.3, 0.6)
fig, axes = plt.subplots(4, 3, figsize=(8, 10))
for row, kind in enumerate(kinds):
    for column, severity in enumerate(severities):
        corrupted = image_corruption(image, kind, severity, seed=42)
        axes[row, column].imshow(corrupted.permute(1, 2, 0))
        axes[row, column].set_title(f"{kind}: {severity}")
        axes[row, column].axis("off")
plt.tight_layout()
plt.show()

## Demonstrate evaluation

Use an initialized baseline to demonstrate the evaluator. Replace it with a validation-selected checkpoint for an actual study.

In [ ]:
with reproducible(42):
    model = RoadNet()
    results = evaluate_corruptions(model, samples=8, size=32, severities=severities)
for result in results:
    print(result)

## Plot and interpret

An untrained model is a negative control. A flat curve can reflect collapse to one class.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for kind in kinds:
    selected = [result for result in results if result.corruption == kind]
    ax.plot(
        [result.severity for result in selected],
        [result.mean_iou for result in selected],
        marker="o",
        label=kind,
    )
ax.set_xlabel("Synthetic corruption severity")
ax.set_ylabel("Test mean IoU, untrained negative control")
ax.set_ylim(0, 1)
ax.legend()
plt.show()

## Research record

Record the seed, configuration, dependency lock and observed failures with each experiment. Results here demonstrate the implemented model under its stated assumptions. They do not establish performance outside this synthetic distribution.